# C3PA QSBC — Standalone CoT Batch Runner (Colab)

Runs the QSBC Phase-1 reasoning generation against a **local** model
(Gemma 4 via Ollama) directly in Colab — no FastAPI/Postgres/opencode
needed. The parsing and prompt construction reuse the repo's exact logic,
so payloads match the API-driven batches byte-for-byte.

**Steps:** clone repo + dataset → parse → build (Sentence, Label, Document)
triples → prompt local model → save JSON → download.

**Knobs at the top of each cell:** `MODEL`, `PER_LABEL`, `DOC_LIMIT`,
`MIN_N`/`MAX_N`, `SEED`, `RUN_ID`, `MAX_WORKERS`.


In [ ]:
# 1. Clone repo + dataset
import os, sys
os.chdir('/content')
if not os.path.exists('/content/qsbc'):
    !git clone -q https://github.com/jpeckenpaugh/qsbc.git qsbc
    !git clone -q https://github.com/MaazBinMusa/C3PA_Dataset.git qsbc/C3PA_Dataset
os.chdir('/content/qsbc')
sys.path.insert(0, '/content/qsbc')
print('cwd:', os.getcwd())
print('repo files:', sorted(os.listdir('.'))[:12])


In [ ]:
# 2. Parse C3PA -> data/*.tsv (exact same pipeline as the local DB)
!python scripts/parse_c3pa.py


In [ ]:
# 3. Install + start Ollama, pull model
import subprocess, time, os, requests

MODEL = 'gemma4:12b'   # change as fits your GPU tier (E2B/E4B/12b/26b/31b)

if not os.path.exists('/usr/local/bin/ollama'):
    !curl -fsSL https://ollama.com/install.sh | sh

if not os.popen('pgrep -x ollama').read().strip():
    subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL,
                     stderr=subprocess.DEVNULL)
    for _ in range(30):
        try:
            requests.get('http://localhost:11434/api/tags', timeout=2)
            break
        except Exception:
            time.sleep(2)

!ollama pull {MODEL}
print('model ready:', requests.get('http://localhost:11434/api/tags').json())


In [ ]:
# 4. Load parsed TSVs -> in-memory corpus (COPY-compatible text decoding)
import csv, random
from collections import defaultdict
from app.promptlib import build_payload, render_prompt

def read_tsv(path):
    rows = []
    with open(path, newline='', encoding='utf-8') as fh:
        for row in csv.reader(fh, delimiter='\t'):
            if not row:
                continue
            # undo COPY's backslash unescaping so text matches the local DB
            rows.append([c.replace('\\\\', '\\') for c in row])
    return rows

labels   = {int(i): name for i, name in read_tsv('data/labels.tsv')}
documents = {int(i): {'subset': s, 'doc_key': k}
             for i, s, n, f, k, u in read_tsv('data/documents.tsv')}
sentences = {int(i): (int(d), t) for i, d, t in read_tsv('data/sentences.tsv')}

sentence_labels = defaultdict(set)
for sid, lid in read_tsv('data/sentence_labels.tsv'):
    sentence_labels[int(sid)].add(int(lid))

# single-label only, excluding 'Others'
single = {sid: next(iter(lids)) for sid, lids in sentence_labels.items()
          if len(lids) == 1 and labels[next(iter(lids))] != 'Others'}

# (doc_id, label_id) -> ordered [(sentence_id, text)]  (id == document order)
groups = defaultdict(list)
for sid, lid in single.items():
    did, text = sentences[sid]
    groups[(did, lid)].append((sid, text))
for k in groups:
    groups[k].sort()

print('single-label sentences:', len(single), '| groups:', len(groups))


In [ ]:
# 5. Sample per-label + build locked INPUT payloads
PER_LABEL = 4      # 4 x 12 labels = 48 runs
DOC_LIMIT = 15     # centered window over same-label sentences
MIN_N, MAX_N = 2, 5
SEED = 42
random.seed(SEED)

by_label = defaultdict(list)
for (did, lid), items in groups.items():
    for sid, text in items:
        by_label[lid].append((sid, did, text))

tasks, seen = [], set()
for lid in sorted(by_label):
    got = tries = 0
    while got < PER_LABEL and tries < PER_LABEL * 10:
        tries += 1
        sid, did, text = random.choice(by_label[lid])
        if sid in seen:
            continue
        seen.add(sid)
        ordered = groups[(did, lid)]
        target_index = next(i for i, (s, t) in enumerate(ordered) if s == sid)
        tasks.append({
            'sentence_id': sid,
            'doc_id': did,
            'label_id': lid,
            'label': labels[lid],
            'doc_key': documents[did]['doc_key'],
            'doc_total': len(ordered),
            'payload': build_payload(text, labels[lid],
                                    [t for s, t in ordered], target_index,
                                    DOC_LIMIT),
        })
        got += 1

print('sampled tasks:', len(tasks))
print('sample payload:', json.dumps(tasks[0]['payload'], ensure_ascii=False)[:200])


In [ ]:
# 6. Run the batch against the local model (4 concurrent workers)
import requests, json
from concurrent.futures import ThreadPoolExecutor
from app.extract import extract_json_array

RUN_ID = 'colab-gemma4-01'   # unique per batch; kept on every row
MAX_WORKERS = 4
OLLAMA_URL = 'http://localhost:11434/v1/chat/completions'

def generate(task):
    prompt = render_prompt(task['payload'], min_n=MIN_N, max_n=MAX_N,
                           generalize=True)
    r = requests.post(
        OLLAMA_URL,
        json={'model': MODEL,
              'messages': [{'role': 'user', 'content': prompt}],
              'temperature': 0.05,
              'stream': False},
        timeout=600,
    )
    r.raise_for_status()
    raw = r.json()['choices'][0]['message']['content'].strip()
    status, _ = extract_json_array(raw)
    return {**task, 'run_id': RUN_ID, 'model': MODEL,
            'raw_response': raw, 'parse_status': status}

results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    for i, res in enumerate(pool.map(generate, tasks), 1):
        results.append(res)
        print(f"[{i}/{len(tasks)}] #{res['sentence_id']} "
              f"{res['label'][:35]!r} -> {res['parse_status']} "
              f"({len(res['raw_response'])} chars)")

from collections import Counter
print('parse_status:', dict(Counter(r['parse_status'] for r in results)))


In [ ]:
# 7. Save results to JSON and download
import json
out_path = f'results_{RUN_ID}.json'
with open(out_path, 'w') as fh:
    json.dump(results, fh, indent=2, ensure_ascii=False)
print('saved', out_path, 'with', len(results), 'rows')

from google.colab import files
files.download(out_path)

# Optional: also stash a copy on Drive if mounted
if os.path.exists('/content/drive/MyDrive'):
    !cp {out_path} /content/drive/MyDrive/qsbc_results/
